In [1]:
# Project Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

warnings.filterwarnings("ignore")

In [2]:
# Load Dataset
df = pd.read_csv("../data/raw/raw_donor_data.csv")
raw_df = df.copy()

In [3]:
# Inspect Data: shape, columns, datatypes, memory usage, and missing values
# Dataset Shape
print("Dataset Shape: ")
print(df.shape)

# Column Names
print("\nColumns: ")
print(df.columns.tolist())

# Datatypes
print("\nDatatypes: ")
print(df.dtypes)

# Memory Usage
print("\nTotal Memory Usage: " + f"{df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Missing values
missing_counts = df.isnull().sum()
print("\nColumns Missing Values:")
print(missing_counts[missing_counts > 0].sort_values(ascending=False))

Dataset Shape: 
(34508, 23)

Columns: 
['DonorUniqueId', 'DonorPostalCode', 'DonorAge', 'MaritalStatus', 'GenderIdentity', 'IsMemberFlag', 'IsAlumnusFlag', 'IsParentFlag', 'HasInvolvementFlag', 'WealthRating', 'AcademicDegreeLevel', 'PreferredAddressType', 'HasEmailFlag', 'ConsecutiveDonorYears', 'LastFiscalYearDonation', 'Donation2FiscalYearsAgo', 'Donation3FiscalYearsAgo', 'Donation4FiscalYearsAgo', 'Donation5FiscalYearsAgo', 'CurrentFiscalYearDonation', 'CumulativeDonationAmount', 'DonorDateOfBirth', 'DonorIndicatorFlag.']

Datatypes: 
DonorUniqueId                  int64
DonorPostalCode              float64
DonorAge                       int64
MaritalStatus                    str
GenderIdentity                   str
IsMemberFlag                     str
IsAlumnusFlag                    str
IsParentFlag                     str
HasInvolvementFlag               str
WealthRating                     str
AcademicDegreeLevel              str
PreferredAddressType             str
HasEmailFla

## Initial Dataset Characteristics
- The dataset contains 34,508 rows and 23 columns
- The dataset size is around 7.2 MB
  - The size is relatively manageable
  - This allows full in-memory processing using pandas
- Multiple fields that are expected to be numeric are currently being stored as strings
  - The fields include: 
    - LastFiscalYearDonation
    - Donation2FiscalYearsAgo
    - Donation3FiscalYearsAgo
    - Donation4FiscalYearsAgo
    - Donation5FiscalYearsAgo
    - CurrentFiscalYearDonation
  - These columns will likely require currency symbol removal, comma removal, conversion to numeric datatype
  - Another field that is stored as a string is DonorDateOfBirth which will likely require datetime conversion and validation
  - Several columns that are stored as stings also represent boolean values which may require standardization with binary encoding
    - These fields include:
      - IsMemberFlag
      - IsAlumnusFlag
      - IsParentFlag
      - HasInvolvementFlag
      - HasEmailFlag
      - DonorIndicatorFlag

- Donation-related features span across multiple fiscal years

- Several categorical and demographic columns contain substantial missing values
  - WealthRating is missing 31,799 values
    - This indicates WealthRating is a sparsely populated feature and may require removal or special handling during feature engineering
  - AcademicDegreeLevel is missing 26,902 values
    - The amount of missing values is significant because it may also require removal or special handling during feature engineering
  - MaritalStatus is missing 24,568 values
    - This amount is significant because it indicates most demographic data is incomplete
  - DonorDateOfBirth is missing 21,190 values
    - This demonstrates the limitations with age and date validation

In [4]:
# Sample Rows
df.sample(10, random_state=121)

,DonorUniqueId,DonorPostalCode,DonorAge,MaritalStatus,GenderIdentity,IsMemberFlag,IsAlumnusFlag,IsParentFlag,HasInvolvementFlag,WealthRating,AcademicDegreeLevel,PreferredAddressType,HasEmailFlag,ConsecutiveDonorYears,LastFiscalYearDonation,Donation2FiscalYearsAgo,Donation3FiscalYearsAgo,Donation4FiscalYearsAgo,Donation5FiscalYearsAgo,CurrentFiscalYearDonation,CumulativeDonationAmount,DonorDateOfBirth,DonorIndicatorFlag.
16199,16200,90265.0,45,NaN,Male,N,Y,N,N,NaN,GM,HOME,N,0,$0,$0,$0,$0,$0,$0,0.0,8/16/1973,N
25925,25926,45864.0,26,NaN,Uknown,N,Y,N,Y,NaN,UB,BUSN,Y,1,$0,$0,$0,$0,$0,$0,0.0,1/3/1992,N
12681,12682,90265.0,46,NaN,Female,N,Y,N,N,NaN,NaN,HOME,Y,0,$0,$0,$0,$0,$0,$0,0.0,3/19/1972,N
34020,34021,90265.0,42,NaN,Male,N,N,N,N,NaN,NaN,NaN,N,1,$0,$0,$0,$0,$0,$0,25.0,NaN,Y
32576,32577,90265.0,60,NaN,Female,N,N,N,N,NaN,NaN,HOME,N,0,$0,$100,$100,$0,$0,$0,200.0,12/20/1958,Y
9047,9048,37172.0,42,Married,Female,N,N,N,N,NaN,NaN,HOME,N,0,$0,$0,$0,$0,$0,$200,200.0,NaN,Y
2988,2989,8026.0,42,Single,Uknown,N,N,N,Y,NaN,NaN,HOME,N,0,$0,$0,$0,$0,$150,$0,150.0,NaN,Y
31189,31190,27568.0,64,NaN,Male,N,N,N,N,NaN,NaN,HOME,N,0,$0,$0,$0,$0,$0,$0,0.0,6/15/1954,N
2661,2662,15482.0,56,NaN,Male,N,N,N,N,NaN,NaN,HOME,N,0,$25,$0,$0,$0,$0,$0,25.0,3/17/1962,Y
20668,20669,56366.0,28,Married,Female,N,N,Y,N,NaN,NaN,HOME,Y,0,$0,$0,$0,$0,$0,$0,0.0,9/20/1990,N


In [5]:
# Summary table about datatypes, missing values, and column uniqueness

summary_df = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.values,
    "missing_count": df.isnull().sum().values,
    "missing_pct": (
        df.isnull().sum().values / len(df) * 100
    ),
    "unique_values": [
        df[col].nunique()
        for col in df.columns
    ]
})

summary_df

,column,dtype,missing_count,missing_pct,unique_values
0,DonorUniqueId,int64,0,0.000000,34508
1,DonorPostalCode,float64,91,0.263707,20991
2,DonorAge,int64,0,0.000000,102
3,MaritalStatus,str,24568,71.195085,6
4,GenderIdentity,str,493,1.428654,5
5,IsMemberFlag,str,0,0.000000,1
6,IsAlumnusFlag,str,0,0.000000,2
7,IsParentFlag,str,0,0.000000,2
8,HasInvolvementFlag,str,0,0.000000,2
9,WealthRating,str,31799,92.149646,8


## Summary Table Observations

- DonorUniqueId contains 34,508 unique values, matching the total number of records in the dataset. This confirms that DonorUniqueId is a primary key and will be treated as a unique identifier rather than a predictive feature.
- DonorPostalCode has 20,991 unique values which makes sense as it provides geographic informaiton and will require feature engineering, such as grouping by prefix or target encoding, before modeling.
- Flag variables that represent binary indicators contains no more than 2 unique values. This is consistent with the expected binary classification and will require verification for consistency and determine whether binary encoding is necessary.
  - These columns are: IsMemberFlag, IsAlumnusFlag, IsParentFlag, HasInvolvementFlag, HasEmailFlag, DonorIndicatorFlag
- IsMemberFlag only contains only a single unique value despite having no missing values suggesting that the feature may have zero variance and may provide no predictive value. 
- WealthRating, AcademicDegreeLevel, MaritalStatus, and DonorDateOfBirth exhibit extremely high missing value percentages at 92.15%, 77.96%, 71.2%, and 61.41%, respectively. 

In [6]:
# Categorical Feature Columns
for col in df.columns:
    if df[col].nunique() < 10:
        print(f"\n{'='*50}")
        print(f"Column: {col}")
        print(f"{'='*50}")
        print(df[col].value_counts(dropna=False))


Column: MaritalStatus
MaritalStatus
NaN              24568
Married           6547
Single            3140
Widowed            157
Divorced            84
Separated            9
Never Married        3
Name: count, dtype: int64

Column: GenderIdentity
GenderIdentity
Female     16678
Male       16233
Uknown      1091
NaN          493
Unknown       12
U              1
Name: count, dtype: int64

Column: IsMemberFlag
IsMemberFlag
N    34508
Name: count, dtype: int64

Column: IsAlumnusFlag
IsAlumnusFlag
N    26086
Y     8422
Name: count, dtype: int64

Column: IsParentFlag
IsParentFlag
N    31807
Y     2701
Name: count, dtype: int64

Column: HasInvolvementFlag
HasInvolvementFlag
N    26533
Y     7975
Name: count, dtype: int64

Column: WealthRating
WealthRating
NaN                      31799
$50,000-$99,999            645
$1-$24,999                 580
$25,000-$49,999            564
$100,000-$249,999          511
$250,000-$499,999          265
$500,000-$999,999           81
$1,000,000-$2,499,999 

## Categorical Feature Observations

- MartialStatus contains a significant amount of missing values with around 71% of records missing a value
  - Among known values, 6,547 donors are married and 3,140 donors are single
- GenderIdentity contains inconsistent category labels
  - These labels are: Female, Male, Uknown, Unknown, and U
  - Uknown, Unknown, and U likely represent the same category; further evaluation is required to determine if these values should be standardize into a single category
- IsMemberFlag only contains a single value: N
  - This means the feature has zero variance
  - The column has no ability to distinguish between donors which suggests:
    - There is a data quality issue
    - A population selection rule
    - Or it is a feature with no predicitve value
  - This likely means that the column will be removed during preprocessing
- AcademicDegreeLevel contains coded values: UB, GM, GP, GD, GC, UC, UG
  - The meaning of these codes are currently unknown. Additional documentation or data dictionary review is required before preprocessing
  

In [7]:
# Stat Summary
df.describe(include="all")

,DonorUniqueId,DonorPostalCode,DonorAge,MaritalStatus,GenderIdentity,IsMemberFlag,IsAlumnusFlag,IsParentFlag,HasInvolvementFlag,WealthRating,AcademicDegreeLevel,PreferredAddressType,HasEmailFlag,ConsecutiveDonorYears,LastFiscalYearDonation,Donation2FiscalYearsAgo,Donation3FiscalYearsAgo,Donation4FiscalYearsAgo,Donation5FiscalYearsAgo,CurrentFiscalYearDonation,CumulativeDonationAmount,DonorDateOfBirth,DonorIndicatorFlag.
count,34508.000000,34417.000000,34508.000000,9940,34015,34508,34508,34508,34508,2709,7606,30465,34508,34508.000000,34508,34508,34508,34508,34508,34508,3.450800e+04,13318,34508
unique,NaN,NaN,NaN,6,5,1,2,2,2,8,7,4,2,NaN,188,189,188,177,182,166,NaN,4812,2
top,NaN,NaN,NaN,Married,Female,N,N,N,N,"$50,000-$99,999",UB,HOME,N,NaN,$0,$0,$0,$0,$0,$0,NaN,9/10/1991,Y
freq,NaN,NaN,NaN,6547,16678,34508,26086,31807,26533,645,3648,28771,23221,NaN,32139,32032,31971,32362,32544,32599,NaN,20,21433
mean,17254.500000,56161.625970,43.366987,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.137475,NaN,NaN,NaN,NaN,NaN,NaN,2.363534e+03,NaN,NaN
std,9961.745881,29966.704506,11.405722,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.423034,NaN,NaN,NaN,NaN,NaN,NaN,1.138014e+05,NaN,NaN
min,1.000000,211.000000,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00,NaN,NaN
25%,8627.750000,29927.000000,42.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00,NaN,NaN
50%,17254.500000,58108.000000,42.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,2.500000e+01,NaN,NaN
75%,25881.250000,90265.000000,42.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,1.440000e+02,NaN,NaN
